# Analyse du Data Drift

Ce notebook constitue le livrable **Analyse du Data Drift**.
Il exécute une comparaison **baseline vs current** à partir des logs de production et génère un rapport HTML Evidently.

In [ ]:
from pathlib import Path
import json
import sys

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'drift' / 'run_drift.py').exists():
            return candidate
    raise FileNotFoundError('Project root not found from current working directory.')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

print('PROJECT_ROOT =', PROJECT_ROOT)

In [ ]:
from drift.run_drift import (
    build_drift_windows,
    build_features_frame,
    ensure_reference_csv,
    load_production_logs,
    run_drift,
)
import pandas as pd

In [ ]:
reports_dir = PROJECT_ROOT / 'reports'
reports_dir.mkdir(parents=True, exist_ok=True)

reference_path = PROJECT_ROOT / 'data' / 'reference' / 'reference.csv'
drift_report_path = reports_dir / 'drift_report_notebook.html'
summary_path = reports_dir / 'monitoring_summary_notebook.json'

ensure_reference_csv(reference_path)
logs = load_production_logs(limit=15000)
baseline_features, current_features, window_meta = build_drift_windows(logs)

if not baseline_features.empty and not current_features.empty:
    drift_summary = run_drift(baseline_features, current_features, drift_report_path)
    drift_summary['reference_source'] = 'logs_baseline_vs_current'
    drift_summary['windowing'] = window_meta
else:
    reference = pd.read_csv(reference_path) if reference_path.exists() else pd.DataFrame()
    production_features = build_features_frame(logs)
    drift_summary = run_drift(reference, production_features, drift_report_path)
    drift_summary['reference_source'] = 'reference_csv_vs_all_logs'
    drift_summary['windowing'] = window_meta

final_summary = {
    'n_logs': int(len(logs)),
    'drift': drift_summary,
}

summary_path.write_text(json.dumps(final_summary, ensure_ascii=False, indent=2), encoding='utf-8')
final_summary

In [ ]:
from IPython.display import display, Markdown

drift = final_summary['drift']
enabled = drift.get('enabled', False)
n_features = drift.get('n_features_compared')
source = drift.get('reference_source')
windowing = drift.get('windowing', {})

display(Markdown(f"""
## Résumé interprétation
- Drift activé: **{enabled}**
- Features comparées: **{n_features}**
- Source de référence: **{source}**
- Fenêtrage: **{windowing}**
- Rapport HTML: **{drift_report_path}**
"""))

if not enabled:
    display(Markdown(f"⚠️ Raison: {drift.get('reason', 'non précisée')}"))

## Livrable produit

- Rapport HTML: `reports/drift_report_notebook.html`
- Résumé JSON: `reports/monitoring_summary_notebook.json`

Ce notebook peut être rejoué pour générer une analyse drift à partir des derniers logs disponibles.